# ChainWeaver — 60-second quick start

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgenio/ChainWeaver/blob/main/notebooks/quickstart.ipynb)

Zero install required — run this top-to-bottom in your browser. You will:

1. Define tools with the `@tool` decorator.
2. Compile them into a typed `Flow` and execute it with **no LLM in the loop**.
3. Export the flow to OpenAI / Anthropic tool specs for your agent.

ChainWeaver's executor is deterministic by design: zero LLM calls, zero network I/O, zero randomness between steps.

In [ ]:
# Install ChainWeaver if it isn't already available (no-op when running
# in an environment that already has it, e.g. CI's editable install).
try:
    import chainweaver  # noqa: F401
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "chainweaver[yaml]"],
        check=True,
    )

## 1. Define tools with `@tool`

The `@tool` decorator introspects type hints to build the Pydantic input schema; the return annotation is the output schema. Decorated tools are plain callables *and* `Tool` objects.

In [ ]:
from pydantic import BaseModel

from chainweaver import tool


class ValueOutput(BaseModel):
    value: int


class FormattedOutput(BaseModel):
    result: str


@tool(description="Doubles a number.")
def double(number: int) -> ValueOutput:
    return {"value": number * 2}


@tool(description="Adds 10 to a value.")
def add_ten(value: int) -> ValueOutput:
    return {"value": value + 10}


@tool(description="Formats a value as a human-readable string.")
def format_result(value: int) -> FormattedOutput:
    return {"result": f"Final value: {value}"}


double(number=5)  # decorated tools are directly callable

## 2. Compile a flow and execute it — zero LLM calls

Wire the tools into a `Flow`, register everything, and run it. Each step is Pydantic-validated; the `ExecutionResult` carries a structured, auditable step log.

In [ ]:
from chainweaver import Flow, FlowExecutor, FlowRegistry, FlowStep

flow = Flow(
    name="double_add_format",
    version="0.1.0",
    description="Doubles a number, adds 10, and formats the result.",
    steps=[
        FlowStep(tool_name="double", input_mapping={"number": "number"}),
        FlowStep(tool_name="add_ten", input_mapping={"value": "value"}),
        FlowStep(tool_name="format_result", input_mapping={"value": "value"}),
    ],
)

registry = FlowRegistry()
registry.register_flow(flow)

executor = FlowExecutor(registry=registry)
executor.register_tool(double)
executor.register_tool(add_ten)
executor.register_tool(format_result)

result = executor.execute_flow("double_add_format", {"number": 5})

print("success     :", result.success)
print("final_output:", result.final_output)
print("\nstep log:")
for step in result.execution_log:
    print(f"  step {step.step_index} {step.tool_name:<13} -> {step.outputs}")

assert result.success
assert result.final_output["result"] == "Final value: 20"

## 3. Export the flow to your agent framework

The flow becomes a single tool your agent can call. ChainWeaver emits plain dict specs — it never imports the `openai` or `anthropic` SDKs.

In [ ]:
import json

from chainweaver.export import flow_to_anthropic_tool, flow_to_openai_function

openai_spec = flow_to_openai_function(flow, executor)
anthropic_spec = flow_to_anthropic_tool(flow, executor)

print("OpenAI function spec:")
print(json.dumps(openai_spec, indent=2))
print("\nAnthropic tool spec:")
print(json.dumps(anthropic_spec, indent=2))